# Exploración de Datos y Resultados — Proyecto Quantum

**Autores:** Nilson David Rivera Gutiérrez (`T00078156`) · Mauro Jose Prasca Bohorquez (`T00065353`)

Este notebook realiza una exploración reproducible (solo lectura) de:
1. `data/experiments/deriv_synthetic_backtest_report.csv` (673 operaciones del simulador).
2. `ml/quantum-slm/quantum_train_alpaca.jsonl` y `ml/quantum-slm/quantum_eval_alpaca.jsonl` (corpus SLM).

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
csv_path = ROOT / 'data' / 'experiments' / 'deriv_synthetic_backtest_report.csv'
df = pd.read_csv(csv_path)
print(f'Total operaciones en simulador: {len(df)}')
df.head()

In [ ]:
# Resumen por instrumento (N, tasa de acierto, neto en R y Profit Factor)
summary = []
for sym, grp in df.groupby('symbol'):
    n = len(grp)
    wins = (grp['r_multiple'] > 0).sum()
    net_r = grp['r_multiple'].sum()
    pos_r = grp.loc[grp['r_multiple'] > 0, 'r_multiple'].sum()
    neg_r = abs(grp.loc[grp['r_multiple'] < 0, 'r_multiple'].sum())
    pf = pos_r / neg_r if neg_r > 0 else float('inf')
    summary.append({'Instrumento': sym, 'N': n, 'Acierto (%)': round(100 * wins / n, 2), 'Neto (R)': round(net_r, 2), 'PF': round(pf, 2)})

pd.DataFrame(summary)

In [ ]:
# Exploración del corpus Alpaca (entrenamiento y evaluación)
train_path = ROOT / 'ml' / 'quantum-slm' / 'quantum_train_alpaca.jsonl'
eval_path = ROOT / 'ml' / 'quantum-slm' / 'quantum_eval_alpaca.jsonl'

train_rows = [json.loads(line) for line in train_path.read_text(encoding='utf-8').splitlines() if line.strip()]
eval_rows = [json.loads(line) for line in eval_path.read_text(encoding='utf-8').splitlines() if line.strip()]
print(f'Filas entrenamiento: {len(train_rows)} | Filas evaluación: {len(eval_rows)}')